# Colab LLM Station

適用於 Google Colab 與雲端 GPU 環境之雙推論引擎（vLLM / Ollama）與雙網路穿透部署架構。

---

## 準備工作：設定 Colab Secrets 環境變數（僅需設定一次）

Google Colab 執行個體位於內部 NAT 網路後方，未配置獨立公網 IPv4 位址。使用具備暫態屬性（Ephemeral）的 Tailscale Auth Key，可在虛擬機器啟動時自動完成身分認證並建立網路節點，無需在執行過程中透過瀏覽器手動登入或掃描 QR Code。

### 1. 於 Tailscale Admin Console 產生金鑰
1. 開啟 [Tailscale Admin Console - Settings - Keys](https://login.tailscale.com/admin/settings/keys)。
2. 點擊 **Generate auth key**。
3. 設定金鑰各項參數：
   - **Description**: 輸入識別名稱（例如 `colab-llm-station`）。
   - **Reusable**: **開啟 (Enabled)**。允許同一金鑰在 Colab Runtime 重啟後重複使用。
   - **Ephemeral**: **開啟 (Enabled，重要)**。當 Colab 執行個體中斷連線或關閉時，系統會自動自 Tailnet 裝置清單中移除該節點，避免累積無效離線主機。
   - **Pre-authorized**: **開啟 (Enabled)**。設備連線後自動核准加入網路，無需至後台手動審批。
   - **Tags**（選用）: 若組織 Tailnet ACL 規範需要標籤，可指定標籤（如 `tag:server`）。
4. 點擊 **Generate key** 並複製產生的金鑰字串（格式為 `tskey-auth-...`）。請妥善保存，Tailscale 僅會顯示此金鑰一次。

### 2. 於 Google Colab Secrets 儲存金鑰與環境變數
1. 在 Google Colab 左側工具列點擊 **Secrets** 面板（鑰匙圖示）。
2. 點擊 **Add new secret**（新增密鑰）。
3. 填入設定數值：
   - **名稱 (Name)**：`TAILSCALE_AUTHKEY`（大小寫需完全一致）。
   - **值 (Value)**：貼上剛才複製的 `tskey-auth-...` 金鑰字串。
   - *(選用)* 若欲下載受權限保護之 Hugging Face 模型，可新增 `HF_TOKEN` 並填入存取權杖。
4. 將 **Notebook access**（筆記本存取權限）開關切換為 **開啟 (ON)**，以允許 Python 透過 `google.colab.userdata` 讀取金鑰。

## 步驟 1: 自 GitHub 拉取專案並建立 Tailscale SSH 連線（快速直連）
直接拉取 GitHub 最新程式碼至 Colab 本地空間（`/content/colab-llm-station`）並啟動 Tailscale 使用者空間網路精靈程序。無需掛載 Google Drive 即可於數秒內建立 SSH 連線，極速就緒。

In [ ]:
# ==============================================================================
# 獨立 SSH 啟動儲存格（基於 Tailscale SSH，免密碼、免管理金鑰）
# ==============================================================================
import os
from google.colab import userdata

# 1. 自 Colab Secrets 讀取並注入環境變數
os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. 自 GitHub 複製專案儲存庫（若已存在則自動拉取最新程式碼）
!git clone https://github.com/yixuanw99/colab-llm-station.git /content/colab-llm-station 2>/dev/null || (cd /content/colab-llm-station && git pull)

# 3. 啟動 Tailscale 並啟用原生 SSH 伺服器
!cd /content/colab-llm-station && bash llm.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

print("\n" + "="*60)
print("[OK] Tailscale SSH 連線已就緒。")
print("1. 本機終端機連線指令：")
print("   ssh root@colab-llm-station")
print("\n2. 本機 VS Code (Remote - SSH) 連線：")
print("   按 F1 -> 輸入 Remote-SSH: Connect to Host... -> 輸入 root@colab-llm-station")
print("   （連線後開啟資料夾：/content/colab-llm-station）")
print("="*60)


## 步驟 2: 初始化環境與套件依賴 (可選於 Colab 執行，或於 SSH 連入後執行)
> 說明：安裝鎖定版本之推論引擎（Ollama、vLLM）與系統相依套件。
> 本步驟可直接在 Colab 儲存格執行，亦可於連入本機 SSH 或 VS Code 終端機後直接執行。

In [ ]:
%%bash
cd /content/colab-llm-station

# 安裝完整環境依賴 (vLLM + Ollama + 系統套件)
bash setup.sh

# 選用：僅安裝特定推論引擎
# bash setup.sh vllm    # 僅安裝 vLLM 與系統依賴
# bash setup.sh ollama  # 僅安裝 Ollama 與系統依賴


## 步驟 3: 啟動 LLM 推論引擎並開放 Tailnet 訪問
> 說明：啟動背景推論服務並將內部服務通訊埠對應至 Tailscale 私有網狀網路。

In [ ]:
%%bash
cd /content/colab-llm-station

# 企業級 GPU (A100 / H100): 啟動 vLLM (Port 8000)
bash llm.sh tunnel tailscale serve 8000
bash llm.sh vllm serve Qwen/Qwen2.5-Coder-32B-Instruct

# 標準級 GPU (T4 / L4): 若使用 Ollama，請取消註解以下指令 (Port 11434):
# bash llm.sh tunnel tailscale serve 11434
# bash llm.sh ollama serve


## 步驟 4: 檢視系統與網路狀態

In [ ]:
!cd /content/colab-llm-station && bash llm.sh status
